In [3]:
pip show langchain langchain-core langchain-openai langchain-community

Name: langchain
Version: 0.3.27
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: c:\users\scl\anaconda3\envs\lang\lib\site-packages
Requires: async-timeout, langchain-core, langchain-text-splitters, langsmith, pydantic, PyYAML, requests, SQLAlchemy
Required-by: langchain-community
---
Name: langchain-core
Version: 1.2.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\users\scl\anaconda3\envs\lang\lib\site-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-community, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langserve
---
Name: langchain-openai
Version: 1.1.6
Summary: An integration package connecting OpenAI and LangChain
Home-page: https://docs.langchain.com/oss/pyth

In [1]:
pip install -U "langchain==0.3.27" "langchain-core==0.3.79" "langchain-openai==0.3.35" "langchain-community==0.3.27"

  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_openai-0.3.35-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_community-0.3.27-py3-none-any.whl.metadata (2.9 kB)
Using cached langchain-0.3.27-py3-none-any.whl (1.0 MB)
Using cached langchain_core-0.3.79-py3-none-any.whl (449 kB)
Using cached langchain_openai-0.3.35-py3-none-any.whl (75 kB)
Using cached langchain_community-0.3.27-py3-none-any.whl (2.5 MB)

   ---------------------------------------- 0/4 [langchain-core]
   ---------------------------------------- 0/4 [langchain-core]
   -------------------- ------------------- 2/4 [langchain]
   -------------------- ------------------- 2/4 [langchain]
   -------------------- ------------------- 2/4 [langchain]
   -------------------- ------------------- 2/4 [langchain]
   -------------------- ------------------- 2/4 [langchain]
   -------------------- ----------

In [ ]:
import os
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents.tool_calling_agent.base import create_tool_calling_agent
from langchain.agents import AgentExecutor

In [ ]:
os.environ["OPENAI_API_KEY"] = "Your_OpenAI_API_key"

In [ ]:
@tool
def get_course_schedule(course_name: str) -> str:
    """Get the schedule for a given course.""" 
    
    schedule = {
        "AI Architect Mastery": "Mondays and Wednesdays, 6 PM - 8 PM",
        "Data Science Bootcamp": "Tuesdays and Thursdays, 7 PM - 9 PM",
        "Full Stack Development": "Saturdays, 10 AM - 4 PM",
    }

    key = course_name.lower()
    
    return schedule.get(key, "Course not found.")

In [ ]:
@tool
def calculate_discounted_price(original_price: float, discount_percentage: float) -> float:
    """Calculate the discounted price given the original price and discount percentage."""
    
    discount_amount = (discount_percentage / 100) * original_price
    discounted_price = original_price - discount_amount
    
    return round(discounted_price, 2)

In [ ]:
@tool
def recommend_next(topic: str) -> str:
    """Its a real life tool which will help to recommend next steps for a specific learning path."""
    
    t = topic.lower()
    
    if "python" in t:
        return "Next step for python is to learn web development using Django or Flask."
    elif "data science" in t:
        return "Next step for data science is to learn machine learning algorithms and their applications."
    elif "machine learning" in t:
        return "Next step for machine learning is to learn deep learning and neural networks."
    else:
        return "Sorry, I don't have a recommendation for that topic."

In [9]:
tools = [get_course_schedule, calculate_discounted_price, recommend_next]

In [10]:
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("System",
     "You are a helpful teaching assistant that helps students with learning paths and course-related queries.\n"
     "Use the tools to answer questions.\n"
     "Rules:\n"
     "1) If the question is about schedules/batches/timings, use get_course_schedule.\n"
     "2) If the question is about price/discount/GST/splits, use calc.\n"
     "3) If the question is about what to learn next, use recommend_next.\n"
     "4) If the user asks something outside these, respond:\n"
     "   'Sorry, I can only help with course schedules, discounted prices, and learning path recommendations.'\n"
     "If you use a tool, summarize the tool result in a human-readable way."
    ),
    ("Human", "{input}"),
    ("Placeholder", "{agent_scratchpad}"),
])

In [14]:
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

In [15]:
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [17]:
demo_query = "I am learning Python. What should I learn next?"
response = executor.invoke({"input": demo_query})



> Entering new AgentExecutor chain...

Invoking: `recommend_next` with `{'topic': 'Python'}`


next step for python is to learn web development using Django or Flask.Next, you should consider learning web development using frameworks like Django or Flask.

> Finished chain.


In [ ]:
demo_query = "Tell me the schedule for AI Architect Mastery course."
response = executor.invoke({"input": demo_query})



> Entering new AgentExecutor chain...

Invoking: `get_course_schedule` with `{'course_name': 'AI Architect Mastery'}`


Mondays and Wednesdays, 6 PM - 8 PMThe AI Architect Mastery course is scheduled for Mondays and Wednesdays from 6 PM to 8 PM.

> Finished chain.
